In [ ]:
# crossed Andreev reflections in metal-semiconductor junction

import kwant
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable
from typing import Tuple
from matplotlib.ticker import ScalarFormatter
from copy import deepcopy
import cmath
from types import SimpleNamespace

def pm_mod(pm, **kwargs):
    pm_lokalne = deepcopy(pm)
    for variable, value in kwargs.items():
        if value is not None:
            setattr(pm_lokalne, variable, value)
    return pm_lokalne

plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10) 

def eV2au(eV: float):
    return 0.03674932587122423*eV

def nm2au(nm: float):
    return 18.89726133921252*nm

def T2au(B: float):
    return 4.254382E-6*B

d_up = np.array([[1 , 0],
                [0 , 0]], dtype = complex)

d_dn = np.array([[0 , 0],
                [0 , 1]], dtype = complex)

s_0 = np.array([[1 , 0],
                [0 , 1]], dtype = complex)

s_x = np.array([[0 , 1],
                [1 , 0]], dtype = complex)

s_y = np.array([[0 , -1j],
                [1j , 0]], dtype = complex)

s_z = np.array([[1 , 0],
                [0 , -1]], dtype = complex)

def maker_V(pm: SimpleNamespace) -> Callable[[float], float]:

    '''
    pass SimpleNamespace parameters (pm) as:
    
    mu - chemical potential;
    x_jun_1 - Fm/Sc junction position in the system;
    x_jun_2 - Sc/Fm junction position in the system;
    Z - electric potential scaling constant;
    a - electric potential blur constant
    '''

    def V(x: float) -> float:

        V_jun_1 = pm.Z*pm.mu*np.exp(-.5*(x - pm.x_jun_1)**2 / pm.a**2)
        V_jun_2 = pm.Z*pm.mu*np.exp(-.5*(x - pm.x_jun_2)**2 / pm.a**2)

        return V_jun_1 + V_jun_2
    
    return V

In [ ]:
def plot_V(pm: SimpleNamespace, name: str = "_") -> None:

    fig, ax = plt.subplots(figsize=(4,4))
    x = np.linspace(0, pm.L, 200)
    V = maker_V(pm)
    ax.plot(x/nm2au(1.0), np.array([V(x_n)/eV2au(1.0) for x_n in x])/eV2au(1.0), color = 'k')
    ax.set_ylabel(r'$V$ (eV)')
    ax.set_xlabel(r'$x$ (nm)')
    ax.grid(True)
    fig.tight_layout()
    #ax.set_title(r"$\mu $" + f"= {pm.mu}, x_jun = {pm.x_jun}, Z = {pm.Z}, a = {pm.a}")

    plt.savefig(name + ".pdf")
    plt.show()

def disperssion(system: kwant.builder.FiniteSystem | kwant.builder.InfiniteSystem,\
                pm: SimpleNamespace, nr_lead: int,\
                k_max: float, n_k_steps: int) -> Tuple[np.ndarray, np.ndarray]:
    
    '''
    Calculate disperssion relation in a lead (nr_lead).
    '''

    bands = kwant.physics.Bands(system.leads[nr_lead])

    ks = np.linspace(-k_max*pm.dx, k_max*pm.dx, n_k_steps)
    Es = [bands(k) for k in ks]

    return (ks/pm.dx, np.asarray(Es))

def plot_disperssion(disperssion: Tuple[np.ndarray, np.ndarray],
                    y_lim: list, name: str = "_") -> None:
    
    ks, energies = disperssion

    if(y_lim[0] == y_lim[1]):
        y_lim[0] = np.min(energies)/eV2au(1.0)
        y_lim[1] = np.max(energies)/eV2au(1.0)
    
    fig, ax = plt.subplots(figsize=(4,4))
    ax.plot(ks*nm2au(1.0), energies/eV2au(1.0), color = 'k')
    ax.set_xlabel(r'$k_x$ (1/nm)')
    ax.set_ylabel(r'$E$ (eV)')
    ax.set_ylim(y_lim[0], y_lim[1])
    ax.grid(True)

    fig.tight_layout()
    plt.savefig(name + ".pdf")
    plt.show()

def TR_particle_hole(system: kwant.builder.FiniteSystem | kwant.builder.InfiniteSystem,
            energies: np.ndarray) -> SimpleNamespace:

    R_pp, R_ph, R_hp, R_hh = [], [], [], []
    T_pp, T_ph, T_hp, T_hh = [], [], [], []
    
    for energy in energies:
        S = kwant.smatrix(system, energy, in_leads = (0,1), out_leads = (0,1))

        R_pp.append(S.transmission((0, 0), (0, 0)))
        R_ph.append(S.transmission((0, 0), (0, 1)))
        R_hp.append(S.transmission((0, 1), (0, 0)))
        R_hh.append(S.transmission((0, 1), (0, 1)))

        T_pp.append(S.transmission((1, 0), (0, 0)))
        T_ph.append(S.transmission((1, 0), (0, 1)))
        T_hp.append(S.transmission((1, 1), (0, 0)))
        T_hh.append(S.transmission((1, 1), (0, 1)))

    return SimpleNamespace(energies = energies,
                            R_pp = np.asarray(R_pp),
                            R_ph = np.asarray(R_ph),
                            R_hp = np.asarray(R_hp),
                            R_hh = np.asarray(R_hh),
                            T_pp = np.asarray(T_pp),
                            T_ph = np.asarray(T_ph),
                            T_hp = np.asarray(T_hp),
                            T_hh = np.asarray(T_hh))

In [ ]:
def makesystem_fmscfm(pm: SimpleNamespace) -> kwant.builder.FiniteSystem | kwant.builder.InfiniteSystem:
    
    '''
    pass SimpleNamespace parameters (pm) as:
    
    L - length of a system;
    dx - dx of the finite difference chain;
    m - mass of an electron;
    mu - chemical potential;
    x_jun_1 - Fm/Sc junction position in the system;
    x_jun_2 - Sc/Fm junction position in the system;
    P_l - exchange field constant in left part of the ferromagnetic metal;
    P_r - exchange field constant in right part of the ferromagnetic metal;
    Delta - superconducting pairing constant;
    Z - electric potential scaling constant;
    a - electric potential blur constant
    '''

    # (1) kinetic Hamiltonian discretization constant
    t = .5/pm.m/pm.dx**2

    # (2) exchange field h in the ferromagnetic metal map
    def h(x: float):

        if(x <= pm.x_jun_1):
            return pm.P_l*pm.mu
        
        elif(x > pm.x_jun_2):
            return pm.P_r*pm.mu
        
        else:
            return 0.0
    
    # (3) superconducting pairing Δ map
    def Delta(x: float):
        return pm.Delta if pm.x_jun_1 <= x <= pm.x_jun_2 else 0.0
    
    # (4) scattering region map
    def chain(pos):
        x, = pos
        return 0 <= x <= pm.L
    
    # (5) electric potential around the junction
    V = maker_V(pm = pm)
    
    # (6) discretized Hamiltonian onsite element
    def onsite(site) -> np.ndarray:

        x, = site.pos

        onsite_00 = 2*t + V(x) - h(x) - pm.mu
        onsite_01 = Delta(x)
        onsite_10 = Delta(x)
        onsite_11 = -2*t - V(x) - h(x) + pm.mu

        return np.array([[onsite_00, onsite_01],\
                        [onsite_10, onsite_11]])
    
    # (7) discretized Hamiltonian hopping element
    def hopping(*site) -> np.ndarray:

        return np.array([[-t, 0],\
                        [0, t]])
    
    # (8) finite difference method lattice of the system
    lat = kwant.lattice.chain(pm.dx, norbs = 2)

    # (9) scattering region of the system
    sys = kwant.Builder()
    sys[lat.shape(chain, (0,))] = onsite
    sys[(lat.neighbors())] = hopping

    # (10) left (ferromagnetic) lead of the system
    llead = kwant.Builder(kwant.TranslationalSymmetry((-pm.dx,)),
                        conservation_law = np.array([[1, 0],\
                                                    [0, 2]]))
    llead[lat(0)] = onsite(SimpleNamespace(pos = (0,)))
    llead[(lat.neighbors())] =  hopping
    sys.attach_lead(llead)

    # (11) right (also ferromagnetic) lead of the system
    rlead = kwant.Builder(kwant.TranslationalSymmetry((pm.dx,)),
                        conservation_law = np.array([[1, 0],\
                                                    [0, 2]]))
    rlead[lat(0)] = onsite(SimpleNamespace(pos = (pm.L,)))
    rlead[(lat.neighbors())] =  hopping
    sys.attach_lead(rlead)

    # (12) finalizing the system
    return sys.finalized()


In [ ]:
pm = SimpleNamespace(

    L = nm2au(510.0), # length of a system
    dx = nm2au(1), # dx of the finite difference chain
    m = 1, # mass of an electron
    mu = eV2au(10e-3), # chemical potential
    x_jun_1 = nm2au(250.0), # Fm/Sc junction position in the system
    x_jun_2 = nm2au(260.0), # Sc/Fm junction position in the system;
    P_l = 0, # exchange field constant in the ferromagnetic metal
    P_r = 0, # exchange field constant in the ferromagnetic metal
    Delta = eV2au(0.25e-3), # superconducting pairing constant
    Z = 0, # electric potential scaling constant
    a = nm2au(1.0) # electric potential blur constant

)

sys = makesystem_fmscfm(pm)
E = np.linspace(0, eV2au(0.5e-3), 40)
tr = TR_particle_hole(system = sys, energies = E)

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].plot(1000*tr.energies/eV2au(1.0), tr.R_pp, label = r'R$_{ee}$')
ax[0].plot(1000*tr.energies/eV2au(1.0), tr.R_hp, label = r'R$_{he}$')
ax[1].plot(1000*tr.energies/eV2au(1.0), tr.T_pp, label = r'T$_{ee}$')
ax[1].plot(1000*tr.energies/eV2au(1.0), tr.T_hp, label = r'T$_{he}$')
ax[0].grid()
ax[1].grid()
ax[0].legend(frameon = False)
ax[1].legend(frameon = False)
ax[0].set_xlabel("E (meV)")
ax[0].set_ylabel("T/R")
ax[0].set_title(r"L$_{sc}$ = 10 nm")
ax[1].set_xlabel("E (meV)")
ax[1].set_ylabel("T/R")
ax[1].set_title(r"L$_{sc}$ = 10 nm")
fig.tight_layout()
plt.savefig("zad2-0.pdf")
plt.show()


In [ ]:
pm = SimpleNamespace(

    L = nm2au(750.0), # length of a system
    dx = nm2au(1), # dx of the finite difference chain
    m = 1, # mass of an electron
    mu = eV2au(10e-3), # chemical potential
    x_jun_1 = nm2au(250.0), # Fm/Sc junction position in the system
    x_jun_2 = nm2au(500.0), # Sc/Fm junction position in the system;
    P_l = 0, # exchange field constant in the ferromagnetic metal
    P_r = 0, # exchange field constant in the ferromagnetic metal
    Delta = eV2au(0.25e-3), # superconducting pairing constant
    Z = 0, # electric potential scaling constant
    a = nm2au(1.0) # electric potential blur constant

)

sys = makesystem_fmscfm(pm)
E = np.linspace(0, eV2au(0.5e-3), 40)
tr = TR_particle_hole(system = sys, energies = E)

fig, ax = plt.subplots(figsize = (4,4))
ax.plot(1000*tr.energies/eV2au(1.0), tr.R_pp, label = r'R$_{ee}$')
ax.plot(1000*tr.energies/eV2au(1.0), tr.R_hp, label = r'R$_{he}$')
ax.plot(1000*tr.energies/eV2au(1.0), tr.T_pp, label = r'T$_{ee}$')
ax.plot(1000*tr.energies/eV2au(1.0), tr.T_hp, label = r'T$_{he}$')
ax.grid()
ax.legend(frameon = False)
ax.set_xlabel("E (meV)")
ax.set_ylabel("T/R")
ax.set_title(r"L$_{sc}$ = 250 nm")
plt.savefig("zad2-1.pdf")
plt.show()


In [ ]:
pm = SimpleNamespace(

    L = nm2au(510.0), # length of a system
    dx = nm2au(1), # dx of the finite difference chain
    m = 1, # mass of an electron
    mu = eV2au(10e-3), # chemical potential
    x_jun_1 = nm2au(250.0), # Fm/Sc junction position in the system
    x_jun_2 = nm2au(260.0), # Sc/Fm junction position in the system;
    P_l = 0, # exchange field constant in the ferromagnetic metal
    P_r = 0, # exchange field constant in the ferromagnetic metal
    Delta = eV2au(0.25e-3), # superconducting pairing constant
    Z = 0, # electric potential scaling constant
    a = nm2au(1.0) # electric potential blur constant

)

sys = makesystem_fmscfm(pm)
L_sc = np.linspace(0, nm2au(500.0), 40)

R_ee, R_he, T_ee, T_he = [], [], [], []
for l_sc in L_sc:
    sys = makesystem_fmscfm(pm_mod(pm,
                                L = nm2au(500.0) + l_sc,
                                x_jun_2 = nm2au(250.0) + l_sc))
    tr = TR_particle_hole(system = sys,
                        energies = np.array([eV2au(0.1e-3)]))
    
    R_ee.append(tr.R_pp[0])
    R_he.append(tr.R_hp[0])
    T_ee.append(tr.T_pp[0])
    T_he.append(tr.T_hp[0])

fig, ax = plt.subplots(figsize = (4,4))
ax.plot(L_sc/nm2au(1.0), R_ee, label = r'R$_{ee}$')
ax.plot(L_sc/nm2au(1.0), R_he, label = r'R$_{he}$')
ax.plot(L_sc/nm2au(1.0), T_ee, label = r'T$_{ee}$')
ax.plot(L_sc/nm2au(1.0), T_he, label = r'T$_{he}$')
ax.grid()
ax.legend(frameon = False)
ax.set_xlabel(r"L$_{sc}$ (nm)")
ax.set_ylabel("T/R")
ax.set_title(r"E = 0.1 meV")
plt.savefig("zad2-2.pdf")
plt.show()

    

In [ ]:
pm = SimpleNamespace(

    L = nm2au(500.0), # length of a system
    dx = nm2au(1), # dx of the finite difference chain
    m = 1, # mass of an electron
    mu = eV2au(10e-3), # chemical potential
    x_jun_1 = nm2au(250.0), # Fm/Sc junction position in the system
    x_jun_2 = nm2au(260.0), # Sc/Fm junction position in the system
    P_l = 0.995, # exchange field constant in the ferromagnetic metal
    P_r = 0, # exchange field constant in the ferromagnetic metal
    Delta = eV2au(0.25e-3), # superconducting pairing constant
    Z = 0, # electric potential scaling constant
    a = nm2au(1.0) # electric potential blur constant

)

sys = makesystem_fmscfm(pm)
L_sc = np.linspace(0, nm2au(500.0), 40)

R_ee, R_he, T_ee, T_he = [], [], [], []
for l_sc in L_sc:
    sys = makesystem_fmscfm(pm_mod(pm,
                                L = nm2au(500.0) + l_sc,
                                x_jun_2 = nm2au(250.0) + l_sc))
    tr = TR_particle_hole(system = sys,
                        energies = np.array([eV2au(0.1e-3)]))
    
    R_ee.append(tr.R_pp[0])
    R_he.append(tr.R_hp[0])
    T_ee.append(tr.T_pp[0])
    T_he.append(tr.T_hp[0])

fig, ax = plt.subplots(figsize = (4,4))
ax.plot(L_sc/nm2au(1.0), R_ee, label = r'R$_{ee}$')
ax.plot(L_sc/nm2au(1.0), R_he, label = r'R$_{he}$')
ax.plot(L_sc/nm2au(1.0), T_ee, label = r'T$_{ee}$')
ax.plot(L_sc/nm2au(1.0), T_he, label = r'T$_{he}$')
ax.grid()
ax.legend(frameon = False)
ax.set_xlabel(r"L$_{sc}$ (nm)")
ax.set_ylabel("T/R")
ax.set_title(r"E = 0.1 meV")
plt.savefig("zad2-3.pdf")
plt.show()